Сгенерировать последовательности, которые состоят из цифр (от 0 до 9) и задаются следующим образом:
x - последовательность цифр
y1 = x1 
yi = xi + x1
Если yi >= 10 то yi = yi - 10
Научить модель рекуррентной нейронной сети предсказывать yi по xi Использовать: RNN, LSTM, GRU


In [1]:
import torch
import numpy as np

def generate_data(n_samples=10000, seq_len=10):

    X = np.random.randint(0, 10, (n_samples, seq_len))
    Y = np.zeros_like(X)

    for i in range(n_samples):
        x1 = X[i, 0]
        Y[i, 0] = x1
        for j in range(1, seq_len):
            Y[i, j] = (X[i, j] + x1) % 10

    return torch.tensor(X), torch.tensor(Y)

X, Y = generate_data()

In [2]:
import torch.nn.functional as F

X = X.long()
X = F.one_hot(X, num_classes=10).float()

In [3]:
import torch.nn as nn

class RNNModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=10,
            hidden_size=32,
            batch_first=True
        )

        self.fc = nn.Linear(32, 10)

    def forward(self, x):

        out, _ = self.rnn(x)
        out = self.fc(out)

        return out

In [4]:
class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=10,
            hidden_size=32,
            batch_first=True
        )

        self.fc = nn.Linear(32, 10)

    def forward(self, x):

        out, _ = self.lstm(x)
        out = self.fc(out)

        return out

In [5]:
class GRUModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.gru = nn.GRU(
            input_size=10,
            hidden_size=32,
            batch_first=True
        )

        self.fc = nn.Linear(32, 10)

    def forward(self, x):

        out, _ = self.gru(x)
        out = self.fc(out)

        return out

In [6]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = LSTMModel()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):

    total_loss = 0

    for x_batch, y_batch in loader:

        optimizer.zero_grad()
        y_batch = y_batch.long()

        output = model(x_batch)

        loss = criterion(
            output.reshape(-1,10),
            y_batch.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("epoch", epoch, "loss", total_loss)

    

epoch 0 loss 360.8392024040222
epoch 1 loss 358.228892326355
epoch 2 loss 344.7878530025482
epoch 3 loss 308.0686694383621
epoch 4 loss 284.3333261013031
epoch 5 loss 242.24121141433716
epoch 6 loss 183.64587897062302
epoch 7 loss 118.22201138734818
epoch 8 loss 67.01545152068138
epoch 9 loss 38.54690670967102
epoch 10 loss 23.658114030957222
epoch 11 loss 15.688486501574516
epoch 12 loss 11.049807462841272
epoch 13 loss 8.169896367937326
epoch 14 loss 6.255801495164633
epoch 15 loss 4.920958980917931
epoch 16 loss 3.9639897160232067
epoch 17 loss 3.253601599484682
epoch 18 loss 2.709913001395762
epoch 19 loss 2.2870017560198903
epoch 20 loss 1.9478666726499796
epoch 21 loss 1.6747690318152308
epoch 22 loss 1.448174529708922
epoch 23 loss 1.2620184035040438
epoch 24 loss 1.1040864195674658
epoch 25 loss 0.9720633756369352
epoch 26 loss 0.8578704739920795
epoch 27 loss 0.7605303006712347
epoch 28 loss 0.6771624402608722
epoch 29 loss 0.6043428040575236


In [7]:
model.eval()

test_x = torch.tensor([[1,4,7,9,3]])
test_x = F.one_hot(test_x, num_classes=10).float()

pred = model(test_x)
pred = pred.argmax(-1)

print(pred)

tensor([[1, 5, 8, 0, 4]])
